# Task 4 & 5: Vector DB + Retriever

**Goal:** Store embeddings in Chroma, implement similarity search (k=5)

## 1. Setup

In [1]:
import os
import re
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found"

from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain.schema import Document

PDF_DIR = Path("../data/raw_pdfs")
CHROMA_DIR = Path("../data/chroma_db")

print("Setup complete ✅")

Setup complete ✅


## 2. Load Chunks (from Task 2)

In [2]:
def parse_filename(filepath: Path) -> dict:
    """Extract metadata from PDF filename."""
    filename = filepath.stem
    
    category_match = re.match(r'^(waterpurifier|airpurifier|vacuumcleaner)', filename, re.IGNORECASE)
    category = category_match.group(1).lower() if category_match else "unknown"
    
    complexity_match = re.search(r'_(simple|complex)', filename, re.IGNORECASE)
    complexity = complexity_match.group(1).lower() if complexity_match else "unknown"
    
    model_match = re.search(r'([A-Z]{2,3}\d{3,}[A-Z]*)', filename, re.IGNORECASE)
    if model_match:
        model_name = model_match.group(1).upper()
    else:
        model_name = f"{category}_{complexity}"
    
    return {
        "source": filepath.name,
        "category": category,
        "complexity": complexity,
        "model_name": model_name,
    }

def chunk_pdf_with_metadata(filepath: Path, chunk_size: int = 1000, chunk_overlap: int = 200) -> list[Document]:
    """Load PDF and split into LangChain Documents with metadata."""
    base_meta = parse_filename(filepath)
    loader = PDFPlumberLoader(str(filepath))
    pages = loader.load()
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    
    all_docs = []
    for page_doc in pages:
        page_num = page_doc.metadata.get("page", 0) + 1
        page_text = page_doc.page_content
        
        if not page_text.strip():
            continue
        
        page_chunks = splitter.split_text(page_text)
        
        for chunk_idx, chunk_text in enumerate(page_chunks):
            chunk_id = f"{base_meta['model_name']}_p{page_num:03d}_c{chunk_idx:03d}"
            
            metadata = {
                **base_meta,
                "page": page_num,
                "chunk_id": chunk_id,
                "chunk_index": chunk_idx,
                "char_count": len(chunk_text),
            }
            
            doc = Document(page_content=chunk_text, metadata=metadata)
            all_docs.append(doc)
    
    return all_docs

In [3]:
# Load all documents
all_docs = []
for pdf in sorted(PDF_DIR.glob("*.pdf")):
    docs = chunk_pdf_with_metadata(pdf)
    all_docs.extend(docs)
    print(f"{pdf.name}: {len(docs)} docs")

print(f"\nTotal: {len(all_docs)} documents")

airpurifier_complex_MFL69726859_00_190321_00.pdf: 67 docs
airpurifier_simple.pdf: 58 docs
vaccumcleaner_complex.pdf: 69 docs
vaccumcleaner_simple_VC_KOR_MFL68700206_05_220117_00_WEB.pdf: 26 docs
waterpurifier_complex.pdf: 56 docs
waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf: 44 docs

Total: 320 documents


## 3. Create Chroma Vector Store

In [ ]:
# Initialize embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create Chroma vector store (persistent)
vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    persist_directory=str(CHROMA_DIR),
    collection_name="lg_manuals",
)

print(f"Vector store created at {CHROMA_DIR}")
print(f"Collection size: {vectorstore._collection.count()} documents")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store created at ../data/chroma_db
Collection size: 320 documents


## 4. Create Retriever

In [5]:
# Create retriever with k=5
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Retriever created (k=5) ✅")

Retriever created (k=5) ✅


## 5. Test Retrieval

In [6]:
# Test queries
test_queries = [
    "정수기 필터 교체는 어떻게 하나요?",
    "공기청정기 필터 청소 방법",
    "청소기 배터리 충전 시간",
    "Wi-Fi 연결이 안 될 때 어떻게 하나요?",
]

In [7]:
def test_retrieval(query: str):
    """Test retrieval and display results."""
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    
    results = retriever.invoke(query)
    
    for i, doc in enumerate(results):
        meta = doc.metadata
        print(f"\n[{i+1}] {meta.get('chunk_id', 'unknown')}")
        print(f"    Source: {meta.get('source', 'unknown')} | Page: {meta.get('page', '?')}")
        print(f"    Category: {meta.get('category', '?')} | Complexity: {meta.get('complexity', '?')}")
        print(f"    Preview: {doc.page_content[:150]}...")
    
    return results

In [8]:
# Run test queries
for query in test_queries:
    test_retrieval(query)


Query: 정수기 필터 교체는 어떻게 하나요?


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



[1] waterpurifier_complex_p028_c000
    Source: waterpurifier_complex.pdf | Page: 28
    Category: waterpurifier | Complexity: complex
    Preview: 28 관리하기
• 정수 필터를 분리할 때 내부 압력에 의해 약간의 물이
알아두기
떨어질 수 있습니다. 아래에 마른 수건을 놓으세요.
물이 얼 때 물속의 미네랄이 중심축으로 모여 서로
결합하여 대부분 탄산칼슘으로 변하고 얼음 속에서
하얗게 보여요. 또한 얼음이 녹으면, ...

[2] MFL69726859_p037_c000
    Source: airpurifier_complex_MFL69726859_00_190321_00.pdf | Page: 37
    Category: airpurifier | Complexity: complex
    Preview: 필터 청소하기 필터 교체하기
1 제품의 커버를 분리하세요. 1 제품의 커버를 분리하고 필터를 교체하세요.
• 커버 분리 및 장착 방법은 필터 보호 • 커버 분리 및 필터 교체 방법은 필터
비닐과 고정 테이프 제거하기를 보호 비닐과 고정 테이프 제거하기를
참조하세요. 참...

[3] MFL71817002_p020_c000
    Source: waterpurifier_simple_WP_KOR_MFL71817002_03_240524_00_OM_WEB.pdf | Page: 20
    Category: waterpurifier | Complexity: simple
    Preview: 20 관리하기
• 새로운 필터로 교체한 경우에는 필터 세척을 10 필터 사용량을 초기화시킨 후에 냉수 120 ml, 정수 120
취소하지 마세요.
ml, 온수 120 ml를 순서대로 출수하여 버린 후
사용하세요.
주의
• 필터를 정위치에 넣지 않을 경우 출수되는 물의 ...

[4] airpurifier_simple_p037_c000
    Source: airpuri

## 6. Load Existing Vector Store (for future use)

In [ ]:
# To reload an existing vector store:
# vectorstore = Chroma(
#     persist_directory=str(CHROMA_DIR),
#     embedding_function=embeddings,
#     collection_name="lg_manuals",
# )
# retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

## 7. Notes

### Retrieval Quality Assessment

| Query | Correct Category | Issue |
|-------|------------------|-------|
| 정수기 필터 교체 | 3/5 | 2 airpurifier results (wrong product) |
| 공기청정기 필터 청소 | 3/5 | 1 vacuumcleaner, 1 waterpurifier |
| 청소기 배터리 충전 | 4/5 | 1 airpurifier filter (irrelevant garbage) |
| Wi-Fi 연결 안됨 | 5/5 | Cross-category OK (common feature) |

### Baseline Limitations

**LIM-005: No Category Filtering**
- Semantic similarity alone mixes products
- User asks "정수기" but gets airpurifier results

**LIM-006: Common Term Contamination**  
- "필터" exists in all categories → cross-category false matches
- Same for "청소", "교체", "사용" etc.

### Week 4 Improvement Candidates
- Metadata filtering by category
- Hybrid search (semantic + keyword)
- Re-ranking with category boost